This notebook disambiguates word senses in BNC extracted sentences.

In [1]:
%load_ext watermark
%watermark
%watermark --iversions

Last updated: 2026-03-27T10:17:51.171632+11:00

Python implementation: CPython
Python version       : 3.14.0
IPython version      : 9.6.0

Compiler    : Clang 16.0.0 (clang-1600.0.26.6)
OS          : Darwin
Release     : 24.6.0
Machine     : arm64
Processor   : arm
CPU cores   : 10
Architecture: 64bit




In [2]:
!pip install -U langchain langchain-core langchain-google-vertexai

  Attempting uninstall: orjson
    Found existing installation: orjson 3.11.4
    Uninstalling orjson-3.11.4:
      Successfully uninstalled orjson-3.11.4
  Attempting uninstall: langgraph-sdkm╸━━━━━━━━━━━━━━━━━━ 13/24 [pytest-benchmark]
    Found existing installation: langgraph-sdk 0.2.10━━━━━━━━━ 13/24 [pytest-benchmark]
    Uninstalling langgraph-sdk-0.2.10:m━━━━━━━━━━━━━━━━━━ 13/24 [pytest-benchmark]
      Successfully uninstalled langgraph-sdk-0.2.10━━━━━━━━━━━ 13/24 [pytest-benchmark]
  Attempting uninstall: langchain-corem━━━━━━━━━━━━━━━━━━ 13/24 [pytest-benchmark]
    Found existing installation: langchain-core 1.2.14━━━━━━━━ 13/24 [pytest-benchmark]
    Uninstalling langchain-core-1.2.14:━━━━━━━━━━━━━━━━━━ 13/24 [pytest-benchmark]
      Successfully uninstalled langchain-core-1.2.14━━━━━━━━━━ 13/24 [pytest-benchmark]
  Attempting uninstall: langgraph-prebuilt90m╺━━━━━━━━━━━ 17/24 [langchain-core]
    Found existing installation: langgraph-prebuilt 1.0.5━━━━━ 17/24 [langchain-

In [3]:
# you need to have Google Cloud account:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=dnqmlrUkJ4yKQ9FzuqnQDsCNIQZMP2&access_type=offline&code_challenge=j-JApn53l0go4fAMllDQ_QZ0uwKtYSyBhw0XDopMPZw&code_challenge_method=S256


Credentials saved to file: [/Users/temuulenkh/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "features-476502" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


Updates are available for some Google Cloud 

In [18]:
import os, re, json, time, logging
from datetime import datetime
from functools import lru_cache
from typing import Any, Dict, List

import pandas as pd
from tqdm import tqdm
from pathlib import Path

from langchain_google_vertexai import ChatVertexAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [20]:
# you need to change the project name with yours before running
GCP_PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT","features-476502")
LOCATION = os.environ.get("GOOGLE_CLOUD_LOCATION", "us-central1")

MODEL_NAME = "gemini-2.5-pro"

TEMPERATURE = 0.1
TOP_P = 0.9
MAX_RETRIES = 3

if not GCP_PROJECT:
    raise RuntimeError("Set GOOGLE_CLOUD_PROJECT env var to your GCP Project ID.")

llm = ChatVertexAI(
    model=MODEL_NAME,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    project=GCP_PROJECT,
    location=LOCATION,
)


/var/folders/s4/dzqp3h_n4k17cqyfz623c1480000gn/T/ipykernel_84812/1026490232.py:14: DeprecationWarning: Use [`ChatGoogleGenerativeAI`][langchain_google_genai.ChatGoogleGenerativeAI] instead.
  llm = ChatVertexAI(


# Create prompt instructions and the core execution chain for animal domain

In [7]:
FEW_SHOTS = [
    ("cat",
     "mammals",
     "The author's face wears a witch's smile of appreciation.",
     "He is clearly familiar with and fond of that **cat**.",
     "The smile may have come as a surprise to connoisseurs of the pictures of him which have been issued to the world.",
     "ANIMAL",
     "Literal animal."),
    ("chicken",
     "birds",
     "well I don't want mine any more",
     "well you're naughty **chicken**",
     "do, do, do",
     "OTHER",
     "Nickname/insult, not the animal."),
    ("chicken",
     "birds",
     "What did you have?",
     "We had **chicken** curry",
     "Yes.",
     "OTHER",
     "Meat, not the animal."),
    ("cat",
     "mammals",
     "Mm mm.",
     "You wanted a **cat** toy, you better play with it!",
     "Got it!",
     "OTHER",
     "Toy, not the animal."),
    ("mouse",
     "mammals",
     "Free MS-DOS",
     "Free DELL **mouse**",
     "Free technical support hotline",
     "OTHER",
     "Computer device, not the animal."),
]

GUIDELINES = """
Label each TARGET as ANIMAL or OTHER using the target sentence and its surrounding context (the previous and next sentences).
Aim for high precision: it is better to label OTHER than to wrongly label ANIMAL.

You are also given a semantic CATEGORY for each target (e.g., “mammals”, “birds”).
CATEGORY represents the intended literal animal sense in the dataset.

If the context clearly refers to a literal animal that matches the CATEGORY, label ANIMAL.
If the context refers to a literal animal that is not the same as the TARGET (e.g. “elephant seal” when TARGET=elephant), label OTHER.
If the context clearly refers to a non-animal sense (e.g. food, tool, verb, vehicle, organisation, metaphor), label OTHER.
If the context is ambiguous or weakly suggests an animal, label OTHER (prefer precision).

Important: The target word is indicated by ** (double asterisks). If the same word appears elsewhere in the sentence without asterisks, ignore those and classify only the asterisk-marked instance.

Pitfalls (common NON-animal senses → label OTHER):
- place names (“Turkey”, “Guinea”)
- verbs/idioms (“to swallow”, “to rabbit on”)
- sports (“cricket” the sport)
- organisations, pubs, teams, products (“Red Lion”, “Jaguar” the car)
- meat/food (“chicken curry”, “beef stew”)
- devices/toys (“computer mouse”, “cat toy”)
- fictional/anthropomorphic characters (“Paddington Bear”, “Roger Rabbit”)
- metaphors, nicknames, insults (“snake”, “chicken”, “lion of industry”)

Literal animals (e.g., “cat”, “dog”, “sperm whale”, “polar bear”) → ANIMAL

Return ONLY a JSON array of objects, one per target:
{"target": <lowercased>, "rationale": short reason, "label": "ANIMAL"|"OTHER", "confidence": float in [0,1]}
No extra keys, no extra text.
""".strip()

@lru_cache(maxsize=1)
def fewshot_block() -> str:
    parts = []
    for t, c, p, s, n, y, r in FEW_SHOTS:
        item = {
            "target": t.lower(),
            "category": c.lower(),
            "label": y,
            "confidence": 0.97,
            "rationale": r,
        }
        parts.append(
            f"TARGET: {t}\n"
            f"CATEGORY: {c}\n"
            f"PREVIOUS: {p}\n"
            f"SENTENCE: {s}\n"
            f"NEXT: {n}\n"
            f"JSON: {json.dumps([item], ensure_ascii=False)}"
        )
    return "\n\n".join(parts)

prompt_tmpl = ChatPromptTemplate.from_messages([
    ("system",
     "You are a careful word-sense classifier for animal names in the "
     "British National Corpus. Aim for high precision (accuracy > recall)."),
    (
        "user",
        "{guidelines}\n\n"
        "{fewshots}\n\n"
        "TARGETS: [{targets_str}]\n"
        "CATEGORIES: [{categories_str}]\n"
        "PREVIOUS: {previous}\n"
        "NEXT: {next}\n"
        "SENTENCE: {sentence}\n"
        "OUTPUT JSON:"
    )
])

def build_prompt_vars(targets, categories, sentence, prev, next_):
    targets_str = ", ".join(sorted({t.lower().strip() for t in targets if t.strip()}))
    categories_str = ", ".join(sorted({c.lower().strip() for c in categories if c.strip()}))

    return {
        "guidelines": GUIDELINES,
        "fewshots": fewshot_block(),
        "targets_str": targets_str,
        "categories_str": categories_str,
        "previous": str(prev).strip() if prev is not None and str(prev).strip().lower() != "nan" else "(none)",
        "next": str(next_).strip() if next_ is not None and str(next_).strip().lower() != "nan" else "(none)",
        "sentence": sentence.strip(),
    }

def extract_json_list(text: str) -> List[Dict[str, Any]]:
    #print("Raw answer: ", text)
    # Try to find a JSON array; if not, fall back to a single object.
    m = re.search(r'\[.*?\]', text, flags=re.S)
    if not m:
        m2 = re.search(r'\{.*\}', text, flags=re.S)
        if not m2:
            raise ValueError("No JSON found in model output.")
        arr = [json.loads(m2.group(0))]
    else:
        arr = json.loads(m.group(0))

    out: List[Dict[str, Any]] = []
    for obj in arr:
        tgt = str(obj.get("target", "")).lower().strip()
        lab = obj.get("label", "OTHER")
        conf = float(obj.get("confidence", 0.0))
        rat = str(obj.get("rationale", ""))
        if lab not in ("ANIMAL", "OTHER"):
            lab = "OTHER"
        out.append(
            {
                "target": tgt,
                "label": lab,
                "confidence": conf,
                "rationale": rat,
            }
        )
    return out

to_json = RunnableLambda(lambda s: extract_json_list(s))
core_chain = prompt_tmpl | (llm | StrOutputParser() | to_json).with_retry()

# Load animal dataset and disambiguate senses

In [8]:
project_folder = os.getcwd()
if os.path.basename(project_folder) == "preprocessing":
    project_folder = os.path.dirname(project_folder)
    
file_path = Path(project_folder) / "data" / "bnc_matches" / "animals_sampled.csv"
out_long  = Path(project_folder) / "data" / "wsd" / "animals_wsd.csv"
os.makedirs(os.path.dirname(out_long), exist_ok=True)

df = pd.read_csv(file_path)

if "row_id" not in df.columns:
    df = df.reset_index().rename(columns={"index": "row_id"})
    df["row_id"] = df["row_id"].astype(int)

# load previously processed rows
if os.path.exists(out_long):
    processed = pd.read_csv(out_long, usecols=["row_id"]).drop_duplicates()
    processed_uids = set(processed["row_id"])
    print(f"{len(processed_uids)} rows have already processed.")
else:
    processed_uids = set()

# filter unprocessed rows
df_new = df[~df["row_id"].isin(processed_uids)].copy()
print(f"{len(df_new)} new rows to process out of {len(df)} total.")

15169 rows have already processed.
0 new rows to process out of 15169 total.


In [11]:
import os
import time
import pandas as pd
from datetime import datetime
from langchain_core.runnables import RunnableParallel, RunnableLambda

def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]


key2item = {}

main_dataset = df_new
batch_size = 16

total_rows = len(main_dataset)
total_batches = (total_rows + batch_size - 1) // batch_size

print(f"   Starting processing at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"   {total_rows} rows, {total_batches} batches\n")

global_start = time.time()
batch_index = 0

def make_lambda(row_data):
    return RunnableLambda(lambda _inp: core_chain.invoke(row_data))

for raw_batch in chunks(main_dataset, batch_size):

    batch_index += 1
    batch_start = time.time()

    print(f"\n===  Batch {batch_index}/{total_batches} ===")
    print(f"Start time: {datetime.now().strftime('%H:%M:%S')}")
    print(f"Batch size: {len(raw_batch)}")

    batch = []

    # build this batch
    for idx, row in raw_batch.iterrows():

        prompt_vars = build_prompt_vars(
            [row.target_word],
            [row.category],
            row.sentence,
            row.previous_sentence or "",
            row.next_sentence or "",
        )

        batch.append((row.row_id, prompt_vars))

        key2item[str(row.row_id)] = {
            "row_id": row.row_id,
            "doc_type": row.doc_type,
            "file_path": row.file_path,
            "sentence_id": row.sentence_id,
            "previous_sentence": row.previous_sentence,
            "sentence": row.sentence,
            "next_sentence": row.next_sentence,
            "target_word": row.target_word,
            "concept": row.concept,
            "category": row.category,
            "n": row.n,
        }

    # Parallel runnable
    parallel = RunnableParallel(
        **{
            str(row_id): make_lambda(row_data)
            for row_id, row_data in batch
        }
    )

    print("Running parallel LLM calls...")

    batch_result = parallel.invoke({})

    # merge results
    results = []
    for k, v in batch_result.items():
        results.append(key2item[k] | v[0])

    pd.DataFrame(results).to_csv(
        out_long,
        mode='a',
        index=False,
        header=not os.path.exists(out_long),
    )

    # timing
    batch_time = time.time() - batch_start
    elapsed = time.time() - global_start
    remaining = (elapsed / batch_index) * (total_batches - batch_index)

    print(f"Batch {batch_index} complete.")
    print(f"Duration: {batch_time:.2f} sec")
    print(f"Total elapsed: {elapsed/60:.2f} min")
    print(f"Estimated remaining: {remaining/60:.2f} min")
    print(f"Current time: {datetime.now().strftime('%H:%M:%S')}")

print("All done!")
print(f"  Total time: {(time.time() - global_start)/60:.2f} minutes")

   Starting processing at 2026-03-27 10:20:10
   0 rows, 0 batches

All done!
  Total time: 0.00 minutes


# Load dataset on flea and disambiguate senses

In [12]:
file_path = Path(project_folder) / "data" / "bnc_matches" / "animals_sampled_flea.csv"
out_long  = Path(project_folder) / "data" / "wsd" / "animals_wsd_flea.csv"
os.makedirs(os.path.dirname(out_long), exist_ok=True)

df = pd.read_csv(file_path)

if "row_id" not in df.columns:
    df = df.reset_index().rename(columns={"index": "row_id"})
    df["row_id"] = df["row_id"].astype(int)

# load previously processed rows
if os.path.exists(out_long):
    processed = pd.read_csv(out_long, usecols=["row_id"]).drop_duplicates()
    processed_uids = set(processed["row_id"])
    print(f"{len(processed_uids)} rows have already processed.")
else:
    processed_uids = set()

# filter unprocessed rows
df_new = df[~df["row_id"].isin(processed_uids)].copy()
print(f"{len(df_new)} new rows to process out of {len(df)} total.")

41 rows have already processed.
0 new rows to process out of 41 total.


In [13]:
import os
import time
import pandas as pd
from datetime import datetime
from langchain_core.runnables import RunnableParallel, RunnableLambda

def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]


key2item = {}

main_dataset = df_new
batch_size = 16

total_rows = len(main_dataset)
total_batches = (total_rows + batch_size - 1) // batch_size

print(f"   Starting processing at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"   {total_rows} rows, {total_batches} batches\n")

global_start = time.time()
batch_index = 0

def make_lambda(row_data):
    return RunnableLambda(lambda _inp: core_chain.invoke(row_data))

for raw_batch in chunks(main_dataset, batch_size):

    batch_index += 1
    batch_start = time.time()

    print(f"\n===  Batch {batch_index}/{total_batches} ===")
    print(f"Start time: {datetime.now().strftime('%H:%M:%S')}")
    print(f"Batch size: {len(raw_batch)}")

    batch = []

    # build this batch
    for idx, row in raw_batch.iterrows():

        prompt_vars = build_prompt_vars(
            [row.target_word],
            [row.category],
            row.sentence,
            row.previous_sentence or "",
            row.next_sentence or "",
        )

        batch.append((row.row_id, prompt_vars))

        key2item[str(row.row_id)] = {
            "row_id": row.row_id,
            "doc_type": row.doc_type,
            "file_path": row.file_path,
            "sentence_id": row.sentence_id,
            "previous_sentence": row.previous_sentence,
            "sentence": row.sentence,
            "next_sentence": row.next_sentence,
            "target_word": row.target_word,
            "concept": row.concept,
            "category": row.category,
            "n": row.n,
        }

    # Parallel runnable
    parallel = RunnableParallel(
        **{
            str(row_id): make_lambda(row_data)
            for row_id, row_data in batch
        }
    )

    print("Running parallel LLM calls...")

    batch_result = parallel.invoke({})

    # merge results
    results = []
    for k, v in batch_result.items():
        results.append(key2item[k] | v[0])

    pd.DataFrame(results).to_csv(
        out_long,
        mode='a',
        index=False,
        header=not os.path.exists(out_long),
    )

    # timing
    batch_time = time.time() - batch_start
    elapsed = time.time() - global_start
    remaining = (elapsed / batch_index) * (total_batches - batch_index)

    print(f"Batch {batch_index} complete.")
    print(f"Duration: {batch_time:.2f} sec")
    print(f"Total elapsed: {elapsed/60:.2f} min")
    print(f"Estimated remaining: {remaining/60:.2f} min")
    print(f"Current time: {datetime.now().strftime('%H:%M:%S')}")

print("\n All done!")
print(f"Total time: {(time.time() - global_start)/60:.2f} minutes")

   Starting processing at 2026-03-27 10:20:38
   0 rows, 0 batches


 All done!
Total time: 0.00 minutes


# Create prompt instructions and the core execution chain for artifact domain

In [15]:
FEW_SHOTS = [
    ("jeep",
     "vehicles",
     "Robyn declared.",
     "The **jeep**'s not that bad.",
     "Besides,’ she added caustically, ‘I don't believe you know a thing about engines anyway!’",
     "ARTIFACT",
     "Literal vehicle artifact."),
    ("level",
     "tools",
     "‘In these sculptures I tried to make an eye.",
     "I raised the head on a base until the eye  is at eye **level**.",
     "You see an eye.’",
     "OTHER",
     "Abstract sense of level, not the tool."),
    ("top",
     "clothing",
     "We've got a reported-stolen list a mile long, but the media only cares about the sexy stuff.",
     "Van Gogh's pretty much **top** of the list.",
     "Poor sod.",
     "OTHER",
     "Abstract ranking, not the garment."),
    ("club",
     "weapons",
     "‘We've got one coming up.",
     "New Bands Night at the youth **club**.’",
     "‘The youth club?",
     "OTHER",
     "Organisation, not the physical club."),
    ("train",
     "vehicles",
     "Both programmes are the largest of their kind in the country.’",
     "ACET is currently offering speakers to inform, motivate, **train** and support.",
     "Churches wanting a speaker should contact the West London offices on 081 840 7879.",
     "OTHER",
     "Verb ‘train’, not the vehicle."),
]

GUIDELINES = """
Label each TARGET as ARTIFACT or OTHER using the target sentence and its surrounding context (the previous and next sentences).
Aim for high precision: it is better to label OTHER than to wrongly label ARTIFACT.

You are also given a semantic CATEGORY for each target (e.g., “tools”, “weapons”).
CATEGORY represents the intended literal man-made object sense in the dataset.

If the context clearly refers to a literal artifact that matches the CATEGORY (e.g. a weapon sense of “tank” when CATEGORY=weapons), label ARTIFACT.
If the context refers to an artifact sense that does not match the CATEGORY (e.g. “water tank” when CATEGORY=weapons, an airplane sense of “plane” when CATEGORY=tools), label OTHER.
If the context refers to a literal artifact that is not the same as the TARGET (e.g. “car park” when TARGET=car), label OTHER.
If the context clearly refers to any non-artifact sense (e.g. organisation, verb, abstract concept, animal), label OTHER.
If the context is ambiguous or weakly suggests an artifact, label OTHER (prefer precision).

Important: The target word is indicated by ** (double asterisks). If the same word appears elsewhere in the sentence without asterisks, ignore those and classify only the asterisk-marked instance.

Pitfalls (common NON-artifact senses → label OTHER):
- abstract concepts, amounts, degrees (“eye level”, “different level”, “top performance”)
- organisations, groups, clubs (“youth club”, “golf club” the organisation)
- verbs/idioms (“train workers”, “to level the ground”)
- events, services, programmes (“training course”, “support service”)
- animals, people, places, and purely abstract concepts

Literal man-made physical objects (e.g., “car”, “knife”, “microwave oven”, “vacuum cleaner”) → ARTIFACT

Return ONLY a JSON array of objects, one per target:
{"target": <lowercased>, "rationale": short reason, "label": "ARTIFACT"|"OTHER", "confidence": float in [0,1]}
No extra keys, no extra text.
""".strip()

@lru_cache(maxsize=1)
def fewshot_block() -> str:
    parts = []
    for t, c, p, s, n, y, r in FEW_SHOTS:
        item = {
            "target": t.lower(),
            "category": c.lower(),
            "label": y,
            "confidence": 0.97,
            "rationale": r,
        }
        parts.append(
            f"TARGET: {t}\n"
            f"CATEGORY: {c}\n"
            f"PREVIOUS: {p}\n"
            f"SENTENCE: {s}\n"
            f"NEXT: {n}\n"
            f"JSON: {json.dumps([item], ensure_ascii=False)}"
        )
    return "\n\n".join(parts)

prompt_tmpl = ChatPromptTemplate.from_messages([
    ("system",
     "You are a careful word-sense classifier for artifact names in the "
     "British National Corpus. Aim for high precision (accuracy > recall)."),
    (
        "user",
        "{guidelines}\n\n"
        "{fewshots}\n\n"
        "TARGETS: [{targets_str}]\n"
        "CATEGORIES: [{categories_str}]\n"
        "PREVIOUS: {previous}\n"
        "NEXT: {next}\n"
        "SENTENCE: {sentence}\n"
        "OUTPUT JSON:"
    )
])

def build_prompt_vars(targets, categories, sentence, prev, next_):
    targets_str = ", ".join(sorted({t.lower().strip() for t in targets if t.strip()}))
    categories_str = ", ".join(sorted({c.lower().strip() for c in categories if c.strip()}))

    return {
        "guidelines": GUIDELINES,
        "fewshots": fewshot_block(),
        "targets_str": targets_str,
        "categories_str": categories_str,
        "previous": str(prev).strip() if prev is not None and str(prev).strip().lower() != "nan" else "(none)",
        "next": str(next_).strip() if next_ is not None and str(next_).strip().lower() != "nan" else "(none)",
        "sentence": sentence.strip(),
    }

def extract_json_list(text: str) -> List[Dict[str, Any]]:
    # Try to find a JSON array; if not, fall back to a single object.
    m = re.search(r'\[.*?\]', text, flags=re.S)
    if not m:
        m2 = re.search(r'\{.*\}', text, flags=re.S)
        if not m2:
            raise ValueError("No JSON found in model output.")
        arr = [json.loads(m2.group(0))]
    else:
        arr = json.loads(m.group(0))

    out: List[Dict[str, Any]] = []
    for obj in arr:
        tgt = str(obj.get("target", "")).lower().strip()
        lab = obj.get("label", "OTHER")
        conf = float(obj.get("confidence", 0.0))
        rat = str(obj.get("rationale", ""))
        if lab not in ("ARTIFACT", "OTHER"):
            lab = "OTHER"
        out.append(
            {
                "target": tgt,
                "label": lab,
                "confidence": conf,
                "rationale": rat,
            }
        )
    return out

to_json = RunnableLambda(lambda s: extract_json_list(s))
core_chain = prompt_tmpl | (llm | StrOutputParser() | to_json).with_retry()

# Load artifact dataset and disambiguate senses

In [16]:
file_path = Path(project_folder) / "data" / "bnc_matches" / "artifacts_sampled.csv"
out_long  = Path(project_folder) / "data" / "wsd" / "artifacts_wsd.csv"
os.makedirs(os.path.dirname(out_long), exist_ok=True)

df = pd.read_csv(file_path)

if "row_id" not in df.columns:
    df = df.reset_index().rename(columns={"index": "row_id"})
    df["row_id"] = df["row_id"].astype(int)

# load previously processed rows
if os.path.exists(out_long):
    processed = pd.read_csv(out_long, usecols=["row_id"]).drop_duplicates()
    processed_uids = set(processed["row_id"])
    print(f"{len(processed_uids)} rows have already processed.")
else:
    processed_uids = set()

# filter unprocessed rows
df_new = df[~df["row_id"].isin(processed_uids)].copy()
print(f"{len(df_new)} new rows to process out of {len(df)} total.")

20590 rows have already processed.
0 new rows to process out of 20590 total.


In [21]:
import os
import time
import pandas as pd
from datetime import datetime
from langchain_core.runnables import RunnableParallel, RunnableLambda

def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]


key2item = {}

main_dataset = df_new
batch_size = 16

total_rows = len(main_dataset)
total_batches = (total_rows + batch_size - 1) // batch_size

print(f"   Starting processing at {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"   {total_rows} rows, {total_batches} batches\n")

global_start = time.time()
batch_index = 0

def make_lambda(row_data):
    return RunnableLambda(lambda _inp: core_chain.invoke(row_data))

for raw_batch in chunks(main_dataset, batch_size):

    batch_index += 1
    batch_start = time.time()

    print(f"\n===  Batch {batch_index}/{total_batches} ===")
    print(f" Start time: {datetime.now().strftime('%H:%M:%S')}")
    print(f" Batch size: {len(raw_batch)}")

    batch = []

    # build this batch
    for idx, row in raw_batch.iterrows():

        prompt_vars = build_prompt_vars(
            [row.target_word],
            [row.category],
            row.sentence,
            row.previous_sentence or "",
            row.next_sentence or "",
        )

        batch.append((row.row_id, prompt_vars))

        key2item[str(row.row_id)] = {
            "row_id": row.row_id,
            "doc_type": row.doc_type,
            "file_path": row.file_path,
            "sentence_id": row.sentence_id,
            "previous_sentence": row.previous_sentence,
            "sentence": row.sentence,
            "next_sentence": row.next_sentence,
            "target_word": row.target_word,
            "concept": row.concept,
            "category": row.category,
            "n": row.n,
        }

    # Parallel runnable
    parallel = RunnableParallel(
        **{
            str(row_id): make_lambda(row_data)
            for row_id, row_data in batch
        }
    )

    print("Running parallel LLM calls...")

    batch_result = parallel.invoke({})

    # merge results
    results = []
    for k, v in batch_result.items():
        results.append(key2item[k] | v[0])

    pd.DataFrame(results).to_csv(
        out_long,
        mode='a',
        index=False,
        header=not os.path.exists(out_long),
    )

    # timing
    batch_time = time.time() - batch_start
    elapsed = time.time() - global_start
    remaining = (elapsed / batch_index) * (total_batches - batch_index)

    print(f"Batch {batch_index} complete.")
    print(f"Duration: {batch_time:.2f} sec")
    print(f"Total elapsed: {elapsed/60:.2f} min")
    print(f"Estimated remaining: {remaining/60:.2f} min")
    print(f"Current time: {datetime.now().strftime('%H:%M:%S')}")

print("\n All done!")
print(f"Total time: {(time.time() - global_start)/60:.2f} minutes")

   Starting processing at 2026-03-27 10:23:35
   0 rows, 0 batches


 All done!
Total time: 0.00 minutes
